# 04 Small CNN trained from scratch: RGB versus 13 bands

**Purpose.** Train a convolutional neural network from scratch on RGB (3 channels) and on all 13 bands, with the same split, preprocessing, augmentation, settings and seeds. The only difference between the two inputs is the number of channels in the first layer, so a difference in the results comes from the bands.

**Functions.** `src/piece_of_crop/cnn.py` (`SmallCNN`, `build_cnn`), `train.py` (`set_seed`, `fit`, `evaluate_model`, `predictions_table`) and `data.py` (`make_loaders`, `make_eval_loader`). The same training loop is used for the ResNet in notebook 05.

**Reads.** `results/split.csv`, `results/band_stats.csv` and the patch cache `data/eurosat_cache.npy`.

**Saves.** `results/cnn_results.csv`, `results/cnn_history.csv`, six test prediction files `results/predictions/cnn_<input>_seed<n>.csv`, the training curves `results/figures/cnn_<input>_training.png` and the trained weights in `models/` (not in Git; notebook 07 uses them).

## 1. Setup and shared data from notebook 02

All functions come from the installed `piece_of_crop` package. On a Mac the network trains on the GPU through MPS if PyTorch supports it, otherwise on the CPU.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report

from piece_of_crop import (CLASS_NAMES, build_cnn, check_test_order, count_parameters, evaluate_model,
                           find_project_root, fit, get_cache, get_device, make_eval_loader, make_loaders,
                           predictions_table, read_split, set_seed)

SEEDS = (0, 1, 2)
BATCH_SIZE = 128
MAX_EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

project_root = find_project_root()
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"
for folder in (results_dir, figures_dir, predictions_dir, models_dir):
    folder.mkdir(parents=True, exist_ok=True)

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
label_ids = np.arange(len(CLASS_NAMES))
assert np.array_equal(np.sort(split["label"].unique()), label_ids), "Expected labels 0..9."
device = get_device()

print("Project root:", project_root)
print("Device:", device)
print("Patches:", split["split"].value_counts().to_dict())
print("Cache:", cache.shape)

## 2. The network

The architecture follows the CNN of the course notebook (CNN_Class_Competition_2_Food_Classification_Explained.pdf, section 2): three blocks of a 3 x 3 convolution, BatchNorm, ReLU and 2 x 2 max pooling with 32, 64 and 128 filters, then global average pooling and one linear layer with 10 outputs.

- **Convolutions** slide a small filter over the patch, so the same pattern is recognised anywhere in it (lecture_8_DL_for_CV_1_compressed.pdf, slide 27). With `padding=1` the size stays the same, and every pooling halves it: 64 x 64 becomes 32, 16 and 8.
- **BatchNorm** normalises the activations inside the network, which gives "more stable gradients" and "a slight regularization effect" (lecture_6ses.8_intro_to_DL-2.pdf, slide 37).
- **Global average pooling** averages each of the 128 feature maps into one number, so the head has only 128 x 10 weights plus 10 biases.
- **The output is logits.** No softmax is added, because `CrossEntropyLoss` applies it internally (lecture_5_intro_to_DL.pdf, slide 25).

The only change between RGB and 13 bands is the number of input channels of the first convolution: 3 or 13.

In [ ]:
for input_type, channels in (("rgb", 3), ("13band", 13)):
    test_model = build_cnn(input_type)
    assert test_model(torch.zeros(2, channels, 64, 64)).shape == (2, 10)
    print(f"{input_type}: {count_parameters(test_model):,} trainable parameters")
print(test_model)

## 3. Training settings

| Setting | Value | Why |
|---|---|---|
| Loss | cross-entropy on the logits | the loss for multi-class classification; it expects logits (lecture_5_intro_to_DL.pdf, slide 25; CNN course notebook, section 4) |
| Optimizer | AdamW, learning rate 0.001 | the course trains its CNN with Adam at 0.001 (CNN course notebook, section 4). AdamW is Adam with weight decay that "does not accumulate in the momentum nor variance" (external: PyTorch documentation, `torch.optim.AdamW`) |
| Weight decay | 0.0001 | L2 regularisation, "called weight decay in DL", shrinks the weights and makes the model less sensitive to single inputs (lecture_2_introduction_to_ML.pdf, slide 40) |
| Batch size | 128 | as in the course CNN notebook (section 1); the same for RGB and 13 bands |
| Epochs | at most 30, early stopping with patience 5 | early stopping is a regularisation technique of the course (lecture_2_introduction_to_ML.pdf, slide 36), and the course notebook on contrastive learning stops when "the validation loss does not improve" for a set number of epochs (contrastive_learning_notebook.pdf, page 17) |
| Best epoch | lowest validation loss, restored at the end | the test set is used only after this choice (lecture_2_introduction_to_ML.pdf, slide 24) |
| Augmentation | random 90 degree turns and flips, training only | see notebook 02, section 6 |

**Why at most 30 epochs with patience 5.** In an earlier run of this notebook with at most 10 epochs and patience 3, four of six networks were still improving their validation loss when the 10 epochs ran out, and one was stopped after 5 epochs because the validation loss jumped up once. That network ended with a validation loss of 0.356, against 0.205 and 0.223 for the other two seeds of the same input. The spread between seeds then measured when training happened to stop, not the input. With 30 epochs and patience 5, a single bad epoch no longer stops training, and the networks can train until the validation loss really stops falling.

The loop is `fit` in `src/piece_of_crop/train.py`. Each step follows the course order: clear the gradients, forward pass, loss, backward pass, update the weights (CNN course notebook, section 4, "Training order").

## 4. Train RGB and 13-band networks with three seeds

For every seed the data loaders are created again, so the shuffling and augmentation follow that seed. After training, each network is evaluated on its training patches (without augmentation), on validation and on test. The test predictions use the team format from `predictions_table`, the same as notebook 03.

In [ ]:
history_rows = []
result_rows = []

for input_type, n_channels in (("rgb", 3), ("13band", 13)):
    train_eval_loader = make_eval_loader(split, cache, band_stats, input_type, "train", batch_size=BATCH_SIZE)
    for seed in SEEDS:
        print(f"\n=== CNN {input_type}, seed {seed} ===")
        set_seed(seed)
        loaders = make_loaders(split, cache, band_stats, input_type, batch_size=BATCH_SIZE, seed=seed)
        sample_images, sample_labels = next(iter(loaders["train"]))
        assert sample_images.shape[1:] == (n_channels, 64, 64)

        model = build_cnn(input_type, num_classes=len(CLASS_NAMES)).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        history, loss_fn = fit(model, loaders, optimizer, device, MAX_EPOCHS, PATIENCE,
                               run_info={"model": "SmallCNN", "input": input_type, "seed": seed})
        history_rows.extend(history)
        best_epoch = min(history, key=lambda row: row["val_loss"])["epoch"]

        # Best checkpoint selected on the validation set; the test set is used only now.
        torch.save({"state_dict": model.cpu().state_dict(), "input": input_type,
                    "seed": seed, "num_classes": len(CLASS_NAMES)},
                   models_dir / f"cnn_{input_type}_seed{seed}.pt")
        model.to(device)

        for subset, loader in (("train", train_eval_loader), ("val", loaders["val"]), ("test", loaders["test"])):
            outcome = evaluate_model(model, loader, loss_fn, device)
            result_rows.append({
                "model": "SmallCNN", "input": input_type, "seed": seed, "split": subset,
                "accuracy": outcome["accuracy"], "macro_f1": outcome["macro_f1"],
                "loss": outcome["loss"], "n_samples": len(outcome["y_true"]),
                "best_epoch": best_epoch, "epochs_run": len(history),
            })
            if subset == "test":
                check_test_order(split, outcome["y_true"])
                test_files = split.loc[split["split"].eq("test"), "file"]
                predictions_table(test_files, outcome["y_true"], outcome["proba"]).to_csv(
                    predictions_dir / f"cnn_{input_type}_seed{seed}.csv", index=False)

cnn_history = pd.DataFrame(history_rows)
cnn_results = pd.DataFrame(result_rows)
cnn_history.to_csv(results_dir / "cnn_history.csv", index=False)
cnn_results.to_csv(results_dir / "cnn_results.csv", index=False)

## 5. Training curves

Training and validation loss per epoch for each seed (left) and validation accuracy (right). The training loss is measured on augmented patches while the network is still changing during the epoch, so it is not directly comparable with the validation loss; the scores in section 6 use the final network on plain training patches.

In [ ]:
for input_type in ("rgb", "13band"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for seed in SEEDS:
        run = cnn_history[(cnn_history["input"] == input_type) & (cnn_history["seed"] == seed)]
        axes[0].plot(run["epoch"], run["train_loss"], ls="--", alpha=0.6, label=f"train seed {seed}")
        axes[0].plot(run["epoch"], run["val_loss"], label=f"val seed {seed}")
        axes[1].plot(run["epoch"], run["val_accuracy"], label=f"seed {seed}")
    axes[0].set(title=f"CNN {input_type}: loss per epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[1].set(title=f"CNN {input_type}: validation accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(figures_dir / f"cnn_{input_type}_training.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Epochs run and best epoch per network:")
display(cnn_results[cnn_results["split"].eq("test")][["input", "seed", "epochs_run", "best_epoch"]])

## 6. Results: RGB versus 13 bands

Mean and standard deviation over the three seeds, for training, validation and test. The per-class reports show seed 0 only; the other seeds are in the prediction files.

In [ ]:
display(cnn_results.groupby(["input", "split"], sort=False)[["accuracy", "macro_f1"]].agg(["mean", "std"]).round(4))

for input_type in ("rgb", "13band"):
    pred = pd.read_csv(predictions_dir / f"cnn_{input_type}_seed0.csv")
    print(f"\nPer-class test report: {input_type}, seed 0")
    print(classification_report(pred["y_true"], pred["y_pred"], labels=label_ids,
                                target_names=list(CLASS_NAMES), zero_division=0))